In [1]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

SILVER = "business_silver"
GOLD = "dim_business"

df_business = spark.table(SILVER)

df_dim = (
    df_business.select(
        "business_id",
        "name",
        "city",
        "state",
        "stars",
        "review_count",
        "categories",
        "_ingest_ts"
    )
)

if spark.catalog.tableExists(GOLD):
    delta = DeltaTable.forName(spark, GOLD)

    (
        delta.alias("t")
        .merge(
            df_dim.alias("s"),
            "t.business_id = s.business_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    (
        df_dim.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(GOLD)
    )

StatementMeta(, ef9bfeb0-4d6c-4725-b5e4-855a91d76934, 3, Finished, Available, Finished, False)